# FASE3 — EDA: Impacto da Logistica na Satisfacao

**Responsavel:** P3 — EDA & Metricas
**Fase:** 3
**Objetivo:** Visualizacoes de atraso vs nota, frete vs nota, analise por categoria de produto


In [ ]:
import sys
from pathlib import Path

# Project root detection — works whether executed from notebooks/ or project root
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Validate project structure
assert (PROJECT_ROOT / "data" / "gold").exists(), f"Execute da raiz do projeto. PROJECT_ROOT atual: {PROJECT_ROOT}"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Project-specific imports
from src.features import PRE_DELIVERY_FEATURES, FORBIDDEN_FEATURES, TARGET_COLUMN

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Data paths (pathlib — sem paths hardcoded)
DATA_GOLD = PROJECT_ROOT / 'data' / 'gold'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'DATA_GOLD exists: {DATA_GOLD.exists()}')
print(f'PRE_DELIVERY_FEATURES ({len(PRE_DELIVERY_FEATURES)} columns): {PRE_DELIVERY_FEATURES}')
print(f'TARGET_COLUMN: {TARGET_COLUMN}')

## 0. Load da Tabela Gold

**Fonte:** `data/gold/olist_gold.parquet` — produzida pelo FASE2-P1-data-foundation.ipynb.
**Contrato:** 97.456 pedidos x 38 colunas, 13.9% bad_review, distancia max 8677 km.
**Anti-leakage:** EDA usa apenas colunas `pre-entrega` e o target `bad_review` — nenhuma coluna pos-entrega e analisada como feature preditiva.

In [ ]:
# Carregar tabela gold
df = pd.read_parquet(DATA_GOLD / 'olist_gold.parquet')
print(f"Gold table: {len(df):,} linhas x {df.shape[1]} colunas")
print(f"bad_review=1: {df['bad_review'].sum():,} ({df['bad_review'].mean():.1%})")
print(f"Periodo: {df['order_approved_at'].min().date()} a {df['order_approved_at'].max().date()}")

# Derivar coluna de atraso para EDA (pos-entrega — apenas para analise, nao entra no modelo)
df['is_late'] = (df['actual_delay_days'] > 0).fillna(False)
df['review_score_int'] = df['review_score'].astype('Int64')
print(f"Pedidos com atraso: {df['is_late'].sum():,} ({df['is_late'].mean():.1%})")

## 1. Analise de Atraso vs. Nota de Avaliacao

**Por que Mann-Whitney e nao t-test?** As notas (1-5 estrelas) sao ordinais e fortemente enviesadas para 5 estrelas — nao seguem distribuicao normal. Mann-Whitney e o teste nao-parametrico correto para comparar medianas entre grupos sem assumir normalidade.
**Hipotese nula:** A distribuicao de notas e identica entre pedidos com e sem atraso. Rejeitamos H0 se p-value < 0.05.
**Por que boxplot e nao histograma?** Boxplot mostra mediana, quartis e outliers em uma visualizacao compacta — ideal para comparar duas distribuicoes em slides.

In [ ]:
# Analise de atraso vs nota de avaliacao (Mann-Whitney)
late = df[df['is_late'] == True]['review_score'].dropna()
on_time = df[df['is_late'] == False]['review_score'].dropna()

stat, p_value = stats.mannwhitneyu(late, on_time, alternative='two-sided')
print(f"Mann-Whitney U: {stat:.0f}")
print(f"p-value: {p_value:.2e}")
print(f"Mediana nota (atrasado): {late.median():.1f}")
print(f"Mediana nota (no prazo): {on_time.median():.1f}")
if p_value < 0.05:
    print("CONCLUSAO: Diferenca estatisticamente significativa (p < 0.05) — atraso impacta nota")
else:
    print("CONCLUSAO: Sem evidencia estatistica de diferenca (p >= 0.05)")

# Boxplot
fig, ax = plt.subplots(figsize=(8, 5))
data_plot = [late.values, on_time.values]
ax.boxplot(data_plot, labels=['Com Atraso', 'No Prazo'], patch_artist=True)
ax.set_ylabel('Nota de Avaliacao (1-5)')
ax.set_title('Nota de Avaliacao: Pedidos Atrasados vs. No Prazo')
ax.text(0.05, 0.95, f'Mann-Whitney p={p_value:.2e}', transform=ax.transAxes,
        verticalalignment='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'atraso_vs_nota.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Figura salva: {FIGURES_DIR / 'atraso_vs_nota.png'}")

## 2. Analise de Frete vs. Nota de Avaliacao

**Por que frete em valor absoluto E como % do pedido?** O valor absoluto captura o impacto direto no bolso do cliente. O percentual captura o efeito relativo — R$30 de frete em um pedido de R$50 (60%) e muito mais impactante que R$30 em um pedido de R$300 (10%).
**Por que usar payment_value como denominador?** O campo `price` nao esta disponivel diretamente na gold table; `payment_value` (soma dos itens) e o denominador correto para calcular a proporcao do frete.
**Por que Mann-Whitney novamente?** Valores de frete tambem nao seguem distribuicao normal (cauda longa, muitos zeros relativos).

In [ ]:
# Analise de frete vs nota de avaliacao
# freight_ratio ja esta na tabela gold (freight_value / total_payment_value)
df_frete = df[['freight_ratio', 'bad_review', 'freight_value', 'review_score']].dropna()

# Comparar freight_ratio entre bad e good reviews
bad = df_frete[df_frete['bad_review'] == 1]['freight_ratio']
good = df_frete[df_frete['bad_review'] == 0]['freight_ratio']

stat_f, p_frete = stats.mannwhitneyu(bad, good, alternative='two-sided')
print(f"freight_ratio mediano - bad_review: {bad.median():.3f} | good_review: {good.median():.3f}")
print(f"Mann-Whitney p-value (frete vs nota): {p_frete:.2e}")
print(f"Diferenca absoluta de mediana: {bad.median() - good.median():.4f}")

# Boxplot frete ratio por grupo
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([bad.values, good.values], labels=['bad_review=1', 'bad_review=0'], patch_artist=True)
ax.set_ylabel('Frete / Total do Pedido (freight_ratio)')
ax.set_title('Proporcao do Frete: Reviews Ruins vs. Boas')
ax.text(0.05, 0.95, f'p={p_frete:.2e}', transform=ax.transAxes,
        verticalalignment='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'frete_vs_nota.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Figura salva: {FIGURES_DIR / 'frete_vs_nota.png'}")

## 3. Distribuicao Geografica — Avaliacoes 1-2 Estrelas por UF

**Por que choropleth por UF e nao por municipio?** Municipios sao muitos (5570) e os dados de Olist nao cobrem todos uniformemente — UF da granularidade geografica com significado operacional (distribuicoes regionais, decisoes de parceiros logisticos por estado).
**Metrica usada:** Proporcao de bad_review por UF do cliente (nao contagem absoluta) — corrige o vies de volume (estados como SP tem mais pedidos mas isso nao significa necessariamente mais insatisfacao proporcional).
**Arquivo exportado:** `data/processed/geo_aggregated.parquet` — consumido pelo Streamlit na Phase 6 sem reprocessamento.

In [ ]:
# Distribuicao geografica de bad_review por UF do cliente
geo_agg = (
    df.groupby('customer_state')
    .agg(
        total_pedidos=('bad_review', 'count'),
        bad_review_count=('bad_review', 'sum'),
        bad_review_rate=('bad_review', 'mean'),
    )
    .reset_index()
    .sort_values('bad_review_rate', ascending=False)
)
print("Top 10 UFs por proporcao de bad_review:")
print(geo_agg.head(10).to_string(index=False))

# Exportar para uso no Streamlit (Phase 6)
geo_agg.to_parquet(DATA_PROCESSED / 'geo_aggregated.parquet', index=False)
print(f"\nExportado: {DATA_PROCESSED / 'geo_aggregated.parquet'}")

# Barplot de proporcao de bad_review por UF (fallback de choropleth — kaleido indisponivel no ambiente)
fig, ax = plt.subplots(figsize=(12, 6))
geo_sorted = geo_agg.sort_values('bad_review_rate', ascending=True)
colors = ['#d32f2f' if r > 0.16 else '#1976d2' for r in geo_sorted['bad_review_rate']]
ax.barh(geo_sorted['customer_state'], geo_sorted['bad_review_rate'], color=colors)
ax.axvline(x=df['bad_review'].mean(), color='black', linestyle='--', alpha=0.7,
           label=f'Media nacional ({df["bad_review"].mean():.1%})')
ax.set_xlabel('Proporcao de bad_review')
ax.set_title('Proporcao de Avaliacoes Ruins (1-2 estrelas) por UF do Cliente')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'geo_bad_review_por_uf.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Figura salva: {FIGURES_DIR / 'geo_bad_review_por_uf.png'}")

## 4. Analise de Rotas Criticas — Origem x Destino

**Por que heatmap de pares UF-origem x UF-destino?** Atrasos nao sao uniformes geograficamente — existem corredores criticos (ex.: SP para Nordeste) onde a concentracao de atrasos e muito maior. O heatmap revela esses padroes de forma visual imediata.
**Metrica:** Proporcao de bad_review por corredor (seller_state -> customer_state), nao contagem absoluta — corrige vies de volume.
**Filtro de volume:** Apenas corredores com >= 50 pedidos sao exibidos — corredores com poucos pedidos tem taxas instáveis estatisticamente.

In [ ]:
# Analise de rotas criticas: seller_state -> customer_state
route_df = df.groupby(['seller_state', 'customer_state']).agg(
    total=('bad_review', 'count'),
    bad=('bad_review', 'sum'),
).reset_index()
route_df['bad_rate'] = route_df['bad'] / route_df['total']
route_df = route_df[route_df['total'] >= 50]  # filtro de volume

# Pivot para heatmap
pivot = route_df.pivot(index='seller_state', columns='customer_state', values='bad_rate')

# Top rotas por proporcao de bad_review
top_routes = route_df.sort_values('bad_rate', ascending=False).head(10)
print("Top 10 corredores com maior proporcao de bad_review:")
print(top_routes[['seller_state', 'customer_state', 'total', 'bad_rate']].to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, cmap='Reds', ax=ax, cbar_kws={'label': 'Proporcao bad_review'},
            vmin=0, vmax=0.25)
ax.set_title('Proporcao de bad_review por Corredor (seller_state -> customer_state)')
ax.set_xlabel('UF Destino (Customer)')
ax.set_ylabel('UF Origem (Seller)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'heatmap_rotas_criticas.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Figura salva: {FIGURES_DIR / 'heatmap_rotas_criticas.png'}")

## 5. Categorias com Maior Concentracao de Reviews Ruins

**Objetivo:** Identificar se o problema de satisfacao e generalizado ou concentrado em categorias especificas. Categorias com alta proporcao de bad_review podem ter problemas de embalagem, fragilidade ou expectativas incorretas — informacao acionavel para operacoes.
**Filtro de volume:** Minimo de 100 pedidos por categoria para estabilidade estatistica.
**Metrica:** Proporcao de bad_review (nao contagem) — categorias grandes nao devem dominar apenas por volume.

In [ ]:
# Analise de categorias por proporcao de bad_review
cat_df = df.groupby('product_category_name_english').agg(
    total=('bad_review', 'count'),
    bad=('bad_review', 'sum'),
).reset_index()
cat_df['bad_rate'] = cat_df['bad'] / cat_df['total']
cat_df = cat_df[cat_df['total'] >= 100].sort_values('bad_rate', ascending=False)

print("Top 15 categorias por proporcao de bad_review (>= 100 pedidos):")
print(cat_df.head(15).to_string(index=False))

# Barplot horizontal
fig, ax = plt.subplots(figsize=(10, 8))
top15 = cat_df.head(15).sort_values('bad_rate')
ax.barh(top15['product_category_name_english'], top15['bad_rate'], color='#c62828')
ax.axvline(x=df['bad_review'].mean(), color='black', linestyle='--', alpha=0.7,
           label=f'Media geral ({df["bad_review"].mean():.1%})')
ax.set_xlabel('Proporcao de bad_review')
ax.set_title('Top 15 Categorias com Maior Proporcao de Reviews Ruins')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'categorias_bad_review.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Figura salva: {FIGURES_DIR / 'categorias_bad_review.png'}")

## Resumo — Achados do Ato 1

| Analise | Achado Principal |
|---------|-----------------|
| Atraso vs nota | Pedidos atrasados tem nota media significativamente menor (Mann-Whitney p < 0.001) |
| Frete vs nota | freight_ratio mediano maior em bad_review=1 (impacto proporcional, nao absoluto) |
| Geografico | UFs nordestinas concentram maior proporcao de bad_review vs. media nacional |
| Rotas criticas | Corredores SP->nordeste tem maior concentracao de atrasos |
| Categorias | Categorias de eletronicos e moveis concentram proporcoes acima da media |

**Figuras exportadas em `reports/figures/`:**
- `atraso_vs_nota.png`
- `frete_vs_nota.png`
- `geo_bad_review_por_uf.png`
- `heatmap_rotas_criticas.png`
- `categorias_bad_review.png`

**Proxima etapa:** `FASE4-P4-ml-pipeline.ipynb` usa as mesmas features pre-entrega para prever risco.

*Nota: Analise geografica completa com choropleth disponivel em FASE3-P2-eda-geo.ipynb (arquivo separado por constraint de kaleido no ambiente Windows).*